In [ ]:
import os
import rasterio
import matplotlib.pyplot as plt
import numpy as np

# Define the specific directory for 10m resolution bands
r10m_path = '/content/drive/MyDrive/PhD/Знімки/S2B_MSIL2A_20260514T090549_N0512_R050_T35UPR_20260514T113326/S2B_MSIL2A_20260514T090549_N0512_R050_T35UPR_20260514T113326.SAFE/GRANULE/L2A_T35UPR_A047981_20260514T091100/IMG_DATA/R10m'
dem_path = '/content/drive/MyDrive/PhD/Знімки/20260611_33497573_PW5oPDvX/dem.tif'

def get_path(folder, band_suffix, resolution='10m'):
    for f in os.listdir(folder):
        if f.endswith(f'_{band_suffix}_{resolution}.jp2'):
            return os.path.join(folder, f)
    return None

# Band paths
b02_p = get_path(r10m_path, 'B02')
b03_p = get_path(r10m_path, 'B03')
b04_p = get_path(r10m_path, 'B04')
b08_p = get_path(r10m_path, 'B08')
tci_p = get_path(r10m_path, 'TCI')

def read_image(path):
    with rasterio.open(path) as src:
        return src.read(1), src.meta

# Load data
b02, _ = read_image(b02_p)
b03, _ = read_image(b03_p)
b04, meta = read_image(b04_p)
# b08, _ = read_image(b08_p)
#tci, _ = read_image(tci_p)

# Create a composite RGB image
rgb_stack = np.stack([b04, b03, b02])

# Visualization helper for contrast stretching
def normalize(array):
    p2, p98 = np.percentile(array, (2, 98))
    return np.clip((array - p2) / (p98 - p2), 0, 1)

# 1. TCI (Pre-rendered)
# axes[0].imshow(tci)
# axes[0].set_title("TCI (True Color Image)")

# 2. Manual RGB Composite
rgb_vis = np.moveaxis(normalize(rgb_stack), 0, -1)
plt.imshow(rgb_vis)
plt.title("Manual RGB Composite (B04, B03, B02)")

# # 3. NIR Band
# axes[2].imshow(normalize(b08), cmap='gray')
# axes[2].set_title("B08 (Near-Infrared)")

plt.show()

print(f"All bands loaded. RGB stack shape: {rgb_stack.shape}")

del b02, b03, b04